# Qwen2-1.5B + QLoRA: Дообучение переводчика глоссы → русский текст

**Google Colab Pro** | GPU: A100 / H100 | Время: ~4–6 ч

## Что делает этот ноутбук
1. Загружает **Qwen/Qwen2-1.5B-Instruct** в 4-битном (QLoRA) режиме
2. Дообучает модель на датасете пар **глосса РЖЯ → русский текст**
   (LoRA rank=16, alpha=32, согласно `params.yaml`)
3. Логирует метрики в **MLflow на DAGsHub**
4. Оценивает BLEU-4 и ROUGE-L на тестовой выборке
5. Сохраняет LoRA-адаптер в Google Drive + DVC push

## Перед запуском
Добавьте в **Colab → Secrets (🔑)**:
- `DAGSHUB_TOKEN` — токен DAGsHub
- `MLFLOW_TRACKING_USERNAME` — логин DAGsHub (`noviyblock`)
- `MLFLOW_TRACKING_PASSWORD` — токен DAGsHub
- `HF_TOKEN` — Hugging Face токен (для скачивания Qwen2)

Датасет (CSV) должен быть в Drive:
```
/content/drive/MyDrive/glossa/data/translations/rsl_gloss_train.csv
/content/drive/MyDrive/glossa/data/translations/rsl_gloss_val.csv
```
Формат CSV: `gloss,translation`

Если файлов нет — ноутбук создаст **синтетический датасет** (ячейка 5).

In [ ]:
# ── 1. Установка зависимостей ─────────────────────────────────────────────────
!pip install -q bitsandbytes>=0.43 peft>=0.10 trl>=0.8 accelerate>=0.29
!pip install -q transformers>=4.40 datasets sacrebleu rouge-score
!pip install -q dagshub mlflow
print('Done')

In [ ]:
# ── 2. Google Drive + Colab Secrets + DAGsHub ────────────────────────────────
import os

from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT   = '/content/drive/MyDrive/glossa'
DATA_DIR     = f'{DRIVE_ROOT}/data/translations'
ADAPTER_DIR  = f'{DRIVE_ROOT}/models/qwen2_lora'
os.makedirs(ADAPTER_DIR, exist_ok=True)

for key in ('DAGSHUB_TOKEN', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD', 'HF_TOKEN'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'[warn] Secret {key} not found')

token = os.environ.get('DAGSHUB_TOKEN') or os.environ.get('MLFLOW_TRACKING_PASSWORD', '')
if token:
    os.environ.setdefault('AWS_ACCESS_KEY_ID', token)
    os.environ.setdefault('AWS_SECRET_ACCESS_KEY', token)
    os.environ.setdefault('MLFLOW_S3_ENDPOINT_URL', 'https://dagshub.com/noviyblock/glossa.s3')

try:
    import dagshub
    dagshub.init(repo_owner='noviyblock', repo_name='glossa', mlflow=True)
    print('[DAGsHub] dagshub.init() OK')
except Exception as e:
    import mlflow
    mlflow.set_tracking_uri('https://dagshub.com/noviyblock/glossa.mlflow')
    print(f'[MLflow] fallback: {e}')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Конфигурация ──────────────────────────────────────────────────────────
import time

CFG = {
    # Model
    'base_model':     'Qwen/Qwen2-1.5B-Instruct',
    'load_in_4bit':   True,
    # LoRA (из params.yaml)
    'lora_r':         16,
    'lora_alpha':     32,
    'lora_dropout':   0.05,
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                            'gate_proj', 'up_proj', 'down_proj'],
    # Training (из params.yaml)
    'batch_size':     4,
    'grad_accum':     4,          # effective batch = 16
    'epochs':         3,
    'lr':             2e-4,
    'warmup_ratio':   0.03,
    'max_seq_length': 512,
    # Eval
    'max_new_tokens': 128,
    'temperature':    0.1,
    # MLflow
    'experiment_name': 'nlp_qwen2_lora_rsl',
    'run_name':         f'qwen2_lora_{time.strftime("%Y%m%d_%H%M")}',
}
print('CFG ready')

In [ ]:
# ── 4. Загрузка модели (4-bit QLoRA) ─────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    CFG['base_model'],
    token=os.environ.get('HF_TOKEN'),
    padding_side='right',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'],
    quantization_config=bnb_cfg,
    device_map='auto',
    token=os.environ.get('HF_TOKEN'),
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

n_params = sum(p.numel() for p in base_model.parameters())
print(f'Загружена модель: {CFG["base_model"]}')
print(f'Параметры: {n_params/1e9:.2f}B')
print(f'dtype: {base_model.dtype}')

In [ ]:
# ── 5. Датасет (загрузка или синтетическая генерация) ─────────────────────────
import pandas as pd
from pathlib import Path
from datasets import Dataset

TRAIN_CSV = f'{DATA_DIR}/rsl_gloss_train.csv'
VAL_CSV   = f'{DATA_DIR}/rsl_gloss_val.csv'


def make_synthetic_data(n: int = 2000) -> pd.DataFrame:
    """Синтетические пары глосс для демонстрации без реального датасета."""
    import random

    TEMPLATES = [
        ('{subj} {verb}',                     '{subj_ru} {verb_ru}.'),
        ('{subj} {verb} {obj}',               '{subj_ru} {verb_ru} {obj_ru}.'),
        ('{subj} {adj} {verb} {obj}',         '{subj_ru} {adj_ru} {verb_ru} {obj_ru}.'),
        ('{subj} {adv} {verb}',               '{subj_ru} {verb_ru} {adv_ru}.'),
        ('ВОПРОС {subj} {verb} {obj}',        '{subj_ru} {verb_ru} {obj_ru}?'),
    ]
    VOCAB = {
        'subj':    [('ЧЕЛОВЕК','человек'), ('МУЖ','мужчина'), ('ЖЕНЩИНА','женщина'),
                    ('РЕБЁНОК','ребёнок'), ('ВРАЧ','врач'), ('УЧИТЕЛЬ','учитель')],
        'verb':    [('ИДТИ','идёт'), ('ГОВОРИТЬ','говорит'), ('СМОТРЕТЬ','смотрит'),
                    ('ПОМОГАТЬ','помогает'), ('УЧИТЬСЯ','учится'), ('РАБОТАТЬ','работает')],
        'obj':     [('КНИГА','книгу'), ('ДОМ','домой'), ('ШКОЛА','в школу'),
                    ('БОЛЬНИЦА','в больницу'), ('МАГАЗИН','в магазин'), ('СЛОВО','слово')],
        'adj':     [('БОЛЬШОЙ','большой'), ('МАЛЕНЬКИЙ','маленький'), ('БЫСТРО','быстро'),
                    ('МЕДЛЕННО','медленно'), ('ХОРОШИЙ','хороший'), ('НОВЫЙ','новый')],
        'adv':     [('БЫСТРО','быстро'), ('МЕДЛЕННО','медленно'), ('СЕЙЧАС','сейчас'),
                    ('ЧАСТО','часто'), ('ВСЕГДА','всегда'), ('НИКОГДА','никогда')],
    }

    rows = []
    for _ in range(n):
        tmpl_g, tmpl_r = random.choice(TEMPLATES)
        mapping = {}
        for key, pairs in VOCAB.items():
            g, r = random.choice(pairs)
            mapping[key] = g
            mapping[f'{key}_ru'] = r
        try:
            gloss = tmpl_g.format(**mapping)
            trans = tmpl_r.format(**mapping)
            rows.append({'gloss': gloss, 'translation': trans})
        except KeyError:
            continue
    return pd.DataFrame(rows)


if Path(TRAIN_CSV).exists() and Path(VAL_CSV).exists():
    df_train = pd.read_csv(TRAIN_CSV)
    df_val   = pd.read_csv(VAL_CSV)
    print(f'Загружено: train={len(df_train)}  val={len(df_val)}')
else:
    print('[warn] Реальный датасет не найден — использую синтетику')
    df_full  = make_synthetic_data(2500)
    split    = int(0.85 * len(df_full))
    df_train = df_full[:split].reset_index(drop=True)
    df_val   = df_full[split:].reset_index(drop=True)
    # Сохраняем для воспроизводимости
    os.makedirs(DATA_DIR, exist_ok=True)
    df_train.to_csv(TRAIN_CSV, index=False)
    df_val.to_csv(VAL_CSV,   index=False)
    print(f'Синтетика: train={len(df_train)}  val={len(df_val)}')

print(df_train.head(3).to_string())

In [ ]:
# ── 6. Форматирование в Chat-формат + подготовка Dataset ─────────────────────
SYSTEM_PROMPT = (
    'Ты — переводчик русского жестового языка (РЖЯ). '
    'Тебе даётся последовательность глосс РЖЯ (записи жестов заглавными буквами). '
    'Переведи её в грамматически правильное русское предложение. '
    'Отвечай только переводом, без пояснений.'
)


def format_example(row: dict) -> str:
    """Преобразует пару глосс/перевод в chat-template Qwen2."""
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Глоссы: {row["gloss"]}'},
        {'role': 'assistant', 'content': row['translation']},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


train_texts = [format_example(r) for r in df_train.to_dict('records')]
val_texts   = [format_example(r) for r in df_val.to_dict('records')]

train_ds = Dataset.from_dict({'text': train_texts})
val_ds   = Dataset.from_dict({'text': val_texts})

print(f'Train примеры: {len(train_ds)}')
print('Пример:\n', train_texts[0][:300], '...')

In [ ]:
# ── 7. LoRA конфигурация + SFTTrainer ────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Подготовка модели к k-bit обучению
base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=CFG['lora_target_modules'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Обучаемые параметры: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)')

training_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    warmup_ratio=CFG['warmup_ratio'],
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=True,
    max_seq_length=CFG['max_seq_length'],
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='paged_adamw_8bit',
    report_to='none',  # MLflow логируем вручную
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
print('SFTTrainer готов')

In [ ]:
# ── 8. Обучение с MLflow ─────────────────────────────────────────────────────
import mlflow

mlflow.set_experiment(CFG['experiment_name'])

with mlflow.start_run(run_name=CFG['run_name']) as run:
    RUN_ID = run.info.run_id
    print(f'MLflow run: {RUN_ID}')

    mlflow.log_params({
        'base_model':     CFG['base_model'],
        'lora_r':         CFG['lora_r'],
        'lora_alpha':     CFG['lora_alpha'],
        'lora_dropout':   CFG['lora_dropout'],
        'batch_size':     CFG['batch_size'],
        'grad_accum':     CFG['grad_accum'],
        'effective_batch': CFG['batch_size'] * CFG['grad_accum'],
        'epochs':         CFG['epochs'],
        'lr':             CFG['lr'],
        'max_seq_length': CFG['max_seq_length'],
        'quantization':   '4bit_nf4',
        'train_samples':  len(train_ds),
        'val_samples':    len(val_ds),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    })

    # Обучение
    train_result = trainer.train()

    # Логируем метрики обучения
    mlflow.log_metrics({
        'train_loss':              train_result.training_loss,
        'train_runtime_s':         train_result.metrics.get('train_runtime', 0),
        'train_samples_per_second': train_result.metrics.get('train_samples_per_second', 0),
    })

    # Validation loss
    eval_metrics = trainer.evaluate()
    mlflow.log_metrics({
        'val_loss': eval_metrics.get('eval_loss', 0),
        'val_ppl':  2 ** eval_metrics.get('eval_loss', 0),  # perplexity
    })

    # Сохранение LoRA-адаптера
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f'Адаптер сохранён: {ADAPTER_DIR}')

    mlflow.set_tag('status', 'trained')
    print(f'train_loss = {train_result.training_loss:.4f}')
    print(f'val_loss   = {eval_metrics.get("eval_loss", 0):.4f}')
    print(f'MLflow run: https://dagshub.com/noviyblock/glossa.mlflow/#/experiments/0/runs/{RUN_ID}')

In [ ]:
# ── 9. Оценка: BLEU-4 + ROUGE-L ──────────────────────────────────────────────
import sacrebleu
from rouge_score import rouge_scorer as rs_module

model.eval()

N_EVAL = min(200, len(df_val))  # ограничение для скорости
df_eval = df_val.sample(N_EVAL, random_state=42).reset_index(drop=True)

refs, hyps = [], []

for _, row in df_eval.iterrows():
    prompt_msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Глоссы: {row["gloss"]}'},
    ]
    prompt = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=CFG['max_new_tokens'],
            temperature=CFG['temperature'],
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(
        ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    refs.append(row['translation'])
    hyps.append(generated)

# BLEU-4
bleu_score = sacrebleu.corpus_bleu(hyps, [refs]).score

# ROUGE-L
scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores = [scorer.score(r, h)['rougeL'].fmeasure for r, h in zip(refs, hyps)]
rouge_l = sum(rouge_scores) / len(rouge_scores)

print(f'BLEU-4:  {bleu_score:.2f}  (target ≥ 35.0)')
print(f'ROUGE-L: {rouge_l:.4f} (target ≥ 0.40)')
print()
for i in range(min(5, N_EVAL)):
    print(f'[{i+1}] Глоссы:  {df_eval.loc[i, "gloss"]}')
    print(f'    Эталон:  {refs[i]}')
    print(f'    Модель:  {hyps[i]}')
    print()

# Логируем в MLflow
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({
        'bleu_4':  bleu_score,
        'rouge_l': rouge_l,
    })
print('Метрики залогированы в MLflow')

In [ ]:
# ── 10. DVC push (опционально) ────────────────────────────────────────────────
import subprocess, shutil

REPO_DIR = '/content/glossa'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/noviyblock/glossa.git', REPO_DIR], check=True)

# Копируем адаптер
target_adapter = f'{REPO_DIR}/models/qwen2_lora'
if os.path.exists(target_adapter):
    shutil.rmtree(target_adapter)
shutil.copytree(ADAPTER_DIR, target_adapter)
print(f'Скопировано: {target_adapter}')

# DVC add + push
for cmd in [
    ['dvc', 'add', 'models/qwen2_lora'],
    ['dvc', 'push'],
]:
    result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(' '.join(cmd), '→', result.returncode)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr[:500])

print('\n=== ИТОГО ===')
print(f'base_model:  {CFG["base_model"]}')
print(f'lora_r:      {CFG["lora_r"]}, alpha={CFG["lora_alpha"]}')
print(f'val_loss:    {eval_metrics.get("eval_loss", 0):.4f}')
print(f'BLEU-4:      {bleu_score:.2f}')
print(f'ROUGE-L:     {rouge_l:.4f}')
print(f'Адаптер:     {ADAPTER_DIR}')
print(f'MLflow UI:   https://dagshub.com/noviyblock/glossa.mlflow')

## Следующие шаги

1. **Если BLEU-4 < 35**: нужен реальный датасет глосс РЖЯ → русский текст.
   Источники: корпус Slovo + ручная разметка переводов, ИИ-аннотирование через GPT-4.

2. **Интеграция**: адаптер загружается в сервис `services/nlp_translation/`:
   ```python
   from peft import PeftModel
   model = AutoModelForCausalLM.from_pretrained(base_model_id, ...)
   model = PeftModel.from_pretrained(model, 'models/qwen2_lora')
   ```

3. **DVC pipeline**: метрики автоматически фиксируются в `reports/nlp_train_metrics.json`.
   После регистрации в MLflow Model Registry — запускается эксперимент `05_nlp_llm_size`.

**Целевые метрики (из params.yaml):**
- BLEU-4 ≥ 35.0
- ROUGE-L ≥ 0.40
- P95 latency ≤ 500 мс